In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/project2025

!unzip -qq "/content/drive/MyDrive/project2025/ErrorIndex.zip"

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
3. 오류 주석 말뭉치/sample_28087.xls:  mismatching "local" filename (3. ьШдыеШ ьг╝ьДЭ ызРынЙь╣Ш/sample_28087.xls),
         continuing with "central" filename version
3. 오류 주석 말뭉치/sample_28087.xml:  mismatching "local" filename (3. ьШдыеШ ьг╝ьДЭ ызРынЙь╣Ш/sample_28087.xml),
         continuing with "central" filename version
3. 오류 주석 말뭉치/sample_28088.xls:  mismatching "local" filename (3. ьШдыеШ ьг╝ьДЭ ызРынЙь╣Ш/sample_28088.xls),
         continuing with "central" filename version
3. 오류 주석 말뭉치/sample_28088.xml:  mismatching "local" filename (3. ьШдыеШ ьг╝ьДЭ ызРынЙь╣Ш/sample_28088.xml),
         continuing with "central" filename version
3. 오류 주석 말뭉치/sample_28089.xls:  mismatching "local" filename (3. ьШдыеШ ьг╝ьДЭ ызРынЙь╣Ш/sample_28089.xls),
         continuing with "central" filename version
3. 오류 주석 말뭉치/sample_28089.xml:  mismatching "local" filename (3. ьШдыеШ ьг╝ьДЭ ызРынЙь╣Ш/sample_28089.xml),
         continuing with "central" filename version
3. 오

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import pandas as pd

xls_paths = glob.glob("/content/drive/MyDrive/project2025/rawErrorIndex/*.xls")
output_dir = "/content/drive/MyDrive/project2025/ErrorIndex"
os.makedirs('/content/drive/MyDrive/project2025/ErrorIndex', exist_ok=True)

for xls_path in xls_paths:

        # 파일 이름만 추출
        base_name = os.path.splitext(os.path.basename(xls_path))[0]

        # 읽기
        df = pd.read_excel(xls_path)

        # 저장 경로 지정
        csv_path = os.path.join(output_dir, base_name + ".csv")

        # CSV 저장
        df.to_csv(csv_path, index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

filename = "/content/drive/MyDrive/project2025/DataSetInfo.xlsx"
df = pd.read_excel(filename)

spokenData = df[(df['구축시기(오류)'].notna()) & (df['자료 유형'] == '구어')]['표본ID']

print(spokenData)

4370      4946
4371      4947
4372      4948
4373      4949
4374      4950
         ...  
34161    37116
34185    37141
34289    37249
34297    37257
34304    37265
Name: 표본ID, Length: 1507, dtype: int64


In [5]:
import pandas as pd
import glob

data_frames = [] # 데이터를 담을 리스트 생성

for i in spokenData:
    df = pd.read_csv(f"/content/drive/MyDrive/project2025/ErrorIndex/sample_{i:05d}.csv")

    # 전체 문장에서 교정할 거리가 있는 경우만 필터링
    filtered_df = df[(df['교정 형태소'].notna()) & (df['문장'] != '문장')][['문장','형태 번호','원 형태소','교정 형태소']]

    # 필터링된 행들을 리스트에 추가
    data_frames.append(filtered_df)
# 결과 출력
for row in data_frames:
    print(row)

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
453                    음 한국게는 친구하고 같이 가면 좋겠어요      2           한국           한국
454                    음 한국게는 친구하고 같이 가면 좋겠어요      3            게            에
455                    음 한국게는 친구하고 같이 가면 좋겠어요      4            는            는

[73 rows x 4 columns]
                               문장  형태 번호       원 형태소      교정 형태소
9    <name3>대학쿄의 한국어과 일 학년 대학생이에요      1  <name3>대학쿄  <name3>대학교
17              <name2>학쿄에서 졸업했어요      1   <name2>학쿄   <name2>학교
24                      넝카이에서 왔어요      1         넝카이         넝카이
32              저는 스스로 한국어르 공부했어요      4         한국어         한국어
33              저는 스스로 한국어르 공부했어요      5           르           를
..                            ...    ...         ...         ...
395                나무가 색을 바구지 않나요      7           않           않
396                나무가 색을 바구지 않나요      8          나요          아요
403             저는 남이섬에 한번 하고 싶어요      6           하           가
417                 우리 가족랑 가고 싶어요      2          가족  

In [ ]:
!pip install ipywidgets  # for vscode
!pip install git+https://git@github.com/SKTBrain/KoBERT.git@master

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import gluonnlp as nlp
import numpy as np
from tqdm.notebook import tqdm
from kobert import get_tokenizer
from kobert import get_pytorch_kobert_model
from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup